<a href="https://colab.research.google.com/github/anujjakhotiya/AI-DL_2026/blob/main/15_10_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""Assignment 7: Behavioral Design Patterns in Python.
Five batch scenarios, each demonstrating Observer, Strategy and Command.
"""

# ============================================================================
# Monday – E-Commerce Website
# ============================================================================
from abc import ABC, abstractmethod

# OBSERVER: product price-drop notifications
class Observer(ABC):
    @abstractmethod
    def update(self, message): pass

class User(Observer):
    def __init__(self, name): self.name = name
    def update(self, message): print(f"Notification to {self.name}: {message}")

class Product:
    def __init__(self, name, price):
        self.name, self.price, self.users = name, price, []
    def subscribe(self, user): self.users.append(user)
    def change_price(self, new_price):
        old = self.price
        self.price = new_price
        if new_price < old:
            for user in self.users:
                user.update(f"{self.name} price dropped: Rs.{old} -> Rs.{new_price}")

# STRATEGY: interchangeable payment methods
class PaymentStrategy(ABC):
    @abstractmethod
    def pay(self, amount): pass
class CardPayment(PaymentStrategy):
    def pay(self, amount): print(f"Paid Rs.{amount} by card")
class UPIPayment(PaymentStrategy):
    def pay(self, amount): print(f"Paid Rs.{amount} by UPI")
class WalletPayment(PaymentStrategy):
    def pay(self, amount): print(f"Paid Rs.{amount} by wallet")
class Checkout:
    def __init__(self, strategy): self.strategy = strategy
    def pay(self, amount): self.strategy.pay(amount)

# COMMAND: cart actions with undo/redo
class Command(ABC):
    @abstractmethod
    def execute(self): pass
    @abstractmethod
    def undo(self): pass
class Cart:
    def __init__(self): self.items, self.coupon = [], None
class AddItem(Command):
    def __init__(self, cart, item): self.cart, self.item = cart, item
    def execute(self): self.cart.items.append(self.item)
    def undo(self): self.cart.items.remove(self.item) if self.item in self.cart.items else None
class ApplyCoupon(Command):
    def __init__(self, cart, coupon): self.cart, self.coupon, self.old = cart, coupon, None
    def execute(self): self.old, self.cart.coupon = self.cart.coupon, self.coupon
    def undo(self): self.cart.coupon = self.old
class CartInvoker:
    def __init__(self): self.history, self.redo_stack = [], []
    def run(self, cmd):
        cmd.execute(); self.history.append(cmd); self.redo_stack.clear()
    def undo(self):
        if self.history:
            cmd = self.history.pop(); cmd.undo(); self.redo_stack.append(cmd)
    def redo(self):
        if self.redo_stack:
            cmd = self.redo_stack.pop(); cmd.execute(); self.history.append(cmd)

p = Product("Headphones", 2000)
p.subscribe(User("Anu")); p.subscribe(User("Riya")); p.change_price(1500)
Checkout(UPIPayment()).pay(1500)
cart, invoker = Cart(), CartInvoker()
invoker.run(AddItem(cart, "Headphones"))
invoker.run(ApplyCoupon(cart, "SAVE10"))
print(cart.items, cart.coupon)
invoker.undo(); print("Undo:", cart.items, cart.coupon)
invoker.redo(); print("Redo:", cart.items, cart.coupon)

# Observer sends price alerts; Strategy switches payment methods; Command stores cart actions and supports undo/redo.


# ============================================================================
# Tuesday – Food Delivery App
# ============================================================================
from abc import ABC, abstractmethod

# OBSERVER: notify agents about new orders
class Observer(ABC):
    @abstractmethod
    def update(self, order): pass
class DeliveryAgent(Observer):
    def __init__(self, name): self.name = name
    def update(self, order): print(f"{self.name} notified about order #{order.order_id}")
class Restaurant:
    def __init__(self, name): self.name, self.agents = name, []
    def subscribe(self, agent): self.agents.append(agent)
    def place_order(self, order):
        print(f"Order #{order.order_id} placed at {self.name}")
        for agent in self.agents: agent.update(order)

# STRATEGY: delivery fee calculation
class DeliveryStrategy(ABC):
    @abstractmethod
    def fee(self, km): pass
class Standard(DeliveryStrategy):
    def fee(self, km): return km * 10
class Express(DeliveryStrategy):
    def fee(self, km): return km * 20 + 20
class FreeCoupon(DeliveryStrategy):
    def fee(self, km): return 0
class DeliveryFee:
    def __init__(self, strategy): self.strategy = strategy
    def calculate(self, km): return self.strategy.fee(km)

# COMMAND: place, cancel and track order
class Command(ABC):
    @abstractmethod
    def execute(self): pass
    @abstractmethod
    def undo(self): pass
class Order:
    def __init__(self, order_id): self.order_id, self.status = order_id, "Created"
class ChangeStatus(Command):
    def __init__(self, order, status):
        self.order, self.status, self.old = order, status, order.status
    def execute(self): self.old, self.order.status = self.order.status, self.status
    def undo(self): self.order.status = self.old
class TrackOrder(Command):
    def __init__(self, order): self.order = order
    def execute(self): print("Current order status:", self.order.status)
    def undo(self): pass
class Invoker:
    def __init__(self): self.history = []
    def run(self, cmd): cmd.execute(); self.history.append(cmd)
    def undo(self):
        if self.history: self.history.pop().undo()

restaurant = Restaurant("Spice Kitchen")
restaurant.subscribe(DeliveryAgent("Aman")); restaurant.subscribe(DeliveryAgent("Neha"))
order = Order(101); restaurant.place_order(order)
print("Express fee for 5 km:", DeliveryFee(Express()).calculate(5))
inv = Invoker(); inv.run(ChangeStatus(order, "Placed")); inv.run(TrackOrder(order))
inv.run(ChangeStatus(order, "Cancelled")); inv.undo(); print("Restored:", order.status)

# Observer alerts delivery agents; Strategy calculates standard/express/free delivery fees; Command models order status actions and undo.


# ============================================================================
# Wednesday – Ride-Hailing App
# ============================================================================
from abc import ABC, abstractmethod

# OBSERVER: notify rider when driver accepts/cancels
class Observer(ABC):
    @abstractmethod
    def update(self, message): pass
class Rider(Observer):
    def __init__(self, name): self.name = name
    def update(self, message): print(f"Alert to {self.name}: {message}")
class Ride:
    def __init__(self, ride_id): self.ride_id, self.observers, self.status = ride_id, [], "Requested"
    def subscribe(self, rider): self.observers.append(rider)
    def set_status(self, status):
        self.status = status
        if status in ("Accepted", "Cancelled"):
            for rider in self.observers: rider.update(f"Ride {self.ride_id} {status.lower()}")

# STRATEGY: fare algorithms
class FareStrategy(ABC):
    @abstractmethod
    def calculate(self, km): pass
class NormalFare(FareStrategy):
    def calculate(self, km): return 40 + 12 * km
class SurgeFare(FareStrategy):
    def calculate(self, km): return (40 + 12 * km) * 1.5
class DiscountFare(FareStrategy):
    def calculate(self, km): return max(20, (40 + 12 * km) * 0.8)
class FareCalculator:
    def __init__(self, strategy): self.strategy = strategy
    def fare(self, km): return self.strategy.calculate(km)

# COMMAND: book, cancel, rate
class Command(ABC):
    @abstractmethod
    def execute(self): pass
    @abstractmethod
    def undo(self): pass
class RideCommand(Command):
    def __init__(self, ride, action):
        self.ride, self.action, self.old = ride, action, ride.status
    def execute(self):
        self.old = self.ride.status
        if self.action in ("Book", "Cancel"): self.ride.set_status("Booked" if self.action == "Book" else "Cancelled")
        else: self.ride.rating = self.action
        print("Action completed:", self.action)
    def undo(self):
        self.ride.set_status(self.old)
        print("Action undone")
class Invoker:
    def run(self, cmd): cmd.execute(); self.last = cmd
    def undo(self):
        if hasattr(self, "last"): self.last.undo()

ride = Ride(501); ride.subscribe(Rider("Anu")); ride.set_status("Accepted")
print("Surge fare for 10 km:", FareCalculator(SurgeFare()).fare(10))
inv = Invoker(); inv.run(RideCommand(ride, "Book")); inv.run(RideCommand(ride, 5)); inv.undo()
inv.run(RideCommand(ride, "Cancel"))

# Observer sends ride-status notifications; Strategy changes fare calculations; Command encapsulates booking, cancellation, and rating.


# ============================================================================
# Thursday – Banking Application
# ============================================================================
from abc import ABC, abstractmethod

# OBSERVER: account balance notifications
class Observer(ABC):
    @abstractmethod
    def update(self, message): pass
class Customer(Observer):
    def __init__(self, name): self.name = name
    def update(self, message): print(f"Alert for {self.name}: {message}")
class Account:
    def __init__(self, number, balance=0):
        self.number, self.balance, self.observers = number, balance, []
    def subscribe(self, customer): self.observers.append(customer)
    def change(self, amount, label):
        self.balance += amount
        for customer in self.observers:
            customer.update(f"{label}; balance Rs.{self.balance}")

# STRATEGY: interest by account type
class InterestStrategy(ABC):
    @abstractmethod
    def calculate(self, balance): pass
class Savings(InterestStrategy):
    def calculate(self, balance): return balance * 0.04
class FixedDeposit(InterestStrategy):
    def calculate(self, balance): return balance * 0.07
class CurrentAccount(InterestStrategy):
    def calculate(self, balance): return 0
class InterestCalculator:
    def __init__(self, strategy): self.strategy = strategy
    def interest(self, balance): return self.strategy.calculate(balance)

# COMMAND: deposit, withdraw and transfer with undo
class Command(ABC):
    @abstractmethod
    def execute(self): pass
    @abstractmethod
    def undo(self): pass
class Deposit(Command):
    def __init__(self, account, amount): self.a, self.amount, self.done = account, amount, False
    def execute(self):
        if self.amount > 0: self.a.change(self.amount, "Deposit"); self.done = True
    def undo(self):
        if self.done: self.a.change(-self.amount, "Undo deposit"); self.done = False
class Withdraw(Command):
    def __init__(self, account, amount): self.a, self.amount, self.done = account, amount, False
    def execute(self):
        if 0 < self.amount <= self.a.balance: self.a.change(-self.amount, "Withdrawal"); self.done = True
        else: print("Withdrawal rejected")
    def undo(self):
        if self.done: self.a.change(self.amount, "Undo withdrawal"); self.done = False
class Transfer(Command):
    def __init__(self, source, target, amount): self.s, self.t, self.amount, self.done = source, target, amount, False
    def execute(self):
        if 0 < self.amount <= self.s.balance:
            self.s.change(-self.amount, "Transfer sent"); self.t.change(self.amount, "Transfer received"); self.done = True
        else: print("Transfer rejected")
    def undo(self):
        if self.done:
            self.t.change(-self.amount, "Undo transfer received"); self.s.change(self.amount, "Undo transfer sent"); self.done = False
class Invoker:
    def __init__(self): self.history = []
    def run(self, cmd): cmd.execute(); self.history.append(cmd)
    def undo(self):
        if self.history: self.history.pop().undo()

a, b = Account("A101", 1000), Account("A102", 500)
a.subscribe(Customer("Anu")); b.subscribe(Customer("Riya"))
inv = Invoker(); inv.run(Deposit(a, 200)); inv.run(Withdraw(a, 100)); inv.undo()
print("Savings interest:", InterestCalculator(Savings()).interest(a.balance))
inv.run(Transfer(a, b, 300)); inv.undo()
print("Final balances:", a.balance, b.balance)

# Observer sends balance alerts; Strategy selects savings/fixed-deposit/current-account interest; Command supports deposit, withdrawal, transfer, and undo.


# ============================================================================
# Friday – Learning Management System (LMS)
# ============================================================================
from abc import ABC, abstractmethod

# OBSERVER: notify students of new courses/assignments
class Observer(ABC):
    @abstractmethod
    def update(self, message): pass
class Student(Observer):
    def __init__(self, name): self.name = name
    def update(self, message): print(f"Notification to {self.name}: {message}")
class Course:
    def __init__(self, title): self.title, self.students = title, []
    def subscribe(self, student): self.students.append(student)
    def publish(self, kind, name):
        for student in self.students:
            student.update(f"New {kind}: {name} in {self.title}")

# STRATEGY: grading methods
class GradingStrategy(ABC):
    @abstractmethod
    def grade(self, marks, maximum=100): pass
class Percentage(GradingStrategy):
    def grade(self, marks, maximum=100): return f"{marks / maximum * 100:.2f}%"
class CGPA(GradingStrategy):
    def grade(self, marks, maximum=100): return round(marks / maximum * 10, 2)
class PassFail(GradingStrategy):
    def grade(self, marks, maximum=100): return "Pass" if marks / maximum >= 0.4 else "Fail"
class GradingService:
    def __init__(self, strategy): self.strategy = strategy
    def calculate(self, marks, maximum=100): return self.strategy.grade(marks, maximum)

# COMMAND: enroll, drop, submit
class Command(ABC):
    @abstractmethod
    def execute(self): pass
    @abstractmethod
    def undo(self): pass
class LMSService:
    def __init__(self): self.courses, self.submissions = [], []
class Enroll(Command):
    def __init__(self, service, course): self.s, self.course = service, course
    def execute(self):
        if self.course not in self.s.courses: self.s.courses.append(self.course)
    def undo(self):
        if self.course in self.s.courses: self.s.courses.remove(self.course)
class Drop(Command):
    def __init__(self, service, course): self.s, self.course, self.index = service, course, None
    def execute(self):
        if self.course in self.s.courses: self.index = self.s.courses.index(self.course); self.s.courses.pop(self.index)
    def undo(self):
        if self.index is not None: self.s.courses.insert(self.index, self.course)
class Submit(Command):
    def __init__(self, service, assignment): self.s, self.assignment = service, assignment
    def execute(self): self.s.submissions.append(self.assignment)
    def undo(self):
        if self.assignment in self.s.submissions: self.s.submissions.remove(self.assignment)
class Invoker:
    def __init__(self): self.history = []
    def run(self, cmd): cmd.execute(); self.history.append(cmd)
    def undo(self):
        if self.history: self.history.pop().undo()

course = Course("Data Structures")
course.subscribe(Student("Anu")); course.subscribe(Student("Rahul"))
course.publish("assignment", "Linked List")
print("Percentage grade:", GradingService(Percentage()).calculate(85))
service, inv = LMSService(), Invoker()
inv.run(Enroll(service, "Data Structures")); inv.run(Submit(service, "Linked List"))
print("Courses:", service.courses, "Submissions:", service.submissions)
inv.undo(); print("After undo:", service.submissions)

# Observer notifies students; Strategy supports percentage, CGPA, and pass/fail grading; Command encapsulates enrollment, dropping, and assignment submission.




Notification to Anu: Headphones price dropped: Rs.2000 -> Rs.1500
Notification to Riya: Headphones price dropped: Rs.2000 -> Rs.1500
Paid Rs.1500 by UPI
['Headphones'] SAVE10
Undo: ['Headphones'] None
Redo: ['Headphones'] SAVE10
Order #101 placed at Spice Kitchen
Aman notified about order #101
Neha notified about order #101
Express fee for 5 km: 120
Current order status: Placed
Restored: Placed
Alert to Anu: Ride 501 accepted
Surge fare for 10 km: 240.0
Action completed: Book
Action completed: 5
Action undone
Alert to Anu: Ride 501 cancelled
Action completed: Cancel
Alert for Anu: Deposit; balance Rs.1200
Alert for Anu: Withdrawal; balance Rs.1100
Alert for Anu: Undo withdrawal; balance Rs.1200
Savings interest: 48.0
Alert for Anu: Transfer sent; balance Rs.900
Alert for Riya: Transfer received; balance Rs.800
Alert for Riya: Undo transfer received; balance Rs.500
Alert for Anu: Undo transfer sent; balance Rs.1200
Final balances: 1200 500
Notification to Anu: New assignment: Linked Lis